# Toy Experiment: Discrete Diffusion (D3PM) — Locally Consistent Patch Mosaics

This notebook extends the continuous toy experiment to **discrete state spaces**.

## What changes vs. the continuous version
| Component | Continuous (`toy_experiment.ipynb`) | Discrete (this notebook) |
|---|---|---|
| Pixel values | ±1 (real-valued) | {0, 1} (binary) |
| Forward process | Add Gaussian noise | Flip pixels via transition matrix Q_t |
| Training loss | MSE on predicted noise | Binary cross-entropy on predicted x₀ |
| Sampling | DDIM (continuous ODE) | D3PM reverse process |
| Image size | 24×24 | 16×16 (faster) |

## What stays the same
- CNN architecture (MinimalResNet with circular padding)
- Black/white toy dataset
- Local consistency evaluation

## Reference
Austin et al. (2021) *Structured Denoising Diffusion Models in Discrete State-Spaces* (D3PM paper).
The forward process used here is **D3PM-uniform** (Section 3.1 / Appendix A.2.1 of that paper).

In [ ]:
import sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader, RandomSampler
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import numpy as np
import csv, os

repo_dir = Path('convolutional_diffusion')
if str(repo_dir) not in sys.path:
    sys.path.append(str(repo_dir))

from models import MinimalResNet

# ── Experiment config ──────────────────────────────────────────────────────────
NUM_LAYERS_LIST = [3, 4, 5, 6, 7, 8]  # sweep to find CNN collapse (L=2 already done)
                                        # RF = 2*(NUM_LAYERS+2)+1
EMB_DIM        = 256
H = W          = 24
NSTEPS_VALID   = 400   # many steps → locally consistent
NSTEPS_INVALID = 3     # few steps  → should show violations
# ──────────────────────────────────────────────────────────────────────────────

K  = 2
T  = 500
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'Config: NUM_LAYERS_LIST={NUM_LAYERS_LIST}, EMB_DIM={EMB_DIM}, H={H}')
print(f'RF formula: 2*(NUM_LAYERS+2)+1  →  '
      f'{[2*(l+2)+1 for l in NUM_LAYERS_LIST]}')
print(f'Sampling: NSTEPS_VALID={NSTEPS_VALID}, NSTEPS_INVALID={NSTEPS_INVALID}')

black = torch.zeros(1, 1, H, W, dtype=torch.long)
white = torch.ones( 1, 1, H, W, dtype=torch.long)
X = torch.cat([black, white], dim=0)
y = torch.tensor([0, 1])
dataset = TensorDataset(X, y)
print(f'Dataset: {len(dataset)} samples, image size {H}x{W}, K={K} categories')

## D3PM Uniform Forward Process

In D3PM-uniform, the forward process at each step independently corrupts each pixel using the transition matrix:

$$Q_t = (1 - \beta_t)\mathbf{I} + \frac{\beta_t}{K} \mathbf{1}\mathbf{1}^T$$

For K=2 (binary), this simply **flips each pixel with probability β_t** at each step.

The cumulative forward process has a closed form:
$$\bar{Q}_t = \bar{\alpha}_t \mathbf{I} + \frac{1 - \bar{\alpha}_t}{K} \mathbf{1}\mathbf{1}^T$$

So sampling $x_t \sim q(x_t | x_0)$ is equivalent to:
- Keep $x_0$ with probability $\bar{\alpha}_t$
- Sample uniformly from $\{0, 1\}$ with probability $1 - \bar{\alpha}_t$

We use a **cosine schedule** for $\bar{\alpha}_t$, matching the existing continuous code.

In [ ]:
def cosine_alpha_bar(t, s=0.008):
    """
    Cosine schedule for alpha_bar_t (signal retention probability).
    t: float tensor in [0, 1], where t=0 means clean data, t=1 means pure noise.
    Returns alpha_bar in [0, 1]: probability that a pixel is unchanged from x_0.
    """
    f_t = torch.cos((t + s) / (1.0 + s) * (torch.pi / 2.0)) ** 2
    f_0 = torch.cos((torch.zeros_like(t) + s) / (1.0 + s) * (torch.pi / 2.0)) ** 2
    return (f_t / f_0.clamp(min=1e-8)).clamp(0.0, 1.0)


def q_sample(x0, t_float):
    """
    Sample x_t ~ q(x_t | x_0) using D3PM-uniform forward process.

    D3PM-uniform cumulative transition (Austin et al. eq. 6 + closed form):
      q(x_t = j | x_0 = i) = alpha_bar_t  if i == j
                            = (1 - alpha_bar_t) / K  otherwise

    Equivalent sampling: keep x0 with prob alpha_bar_t, else sample Uniform({0,...,K-1}).

    x0:      LongTensor [B, 1, H, W] with values in {0, ..., K-1}
    t_float: FloatTensor [B] in (0, 1]
    Returns: LongTensor [B, 1, H, W]
    """
    ab = cosine_alpha_bar(t_float).to(x0.device)  # [B]
    ab = ab[:, None, None, None].expand_as(x0.float())  # [B, 1, H, W]

    # Keep original pixel with probability alpha_bar_t
    keep = torch.bernoulli(ab).bool()
    # Uniform noise over {0, ..., K-1}
    noise = torch.randint(0, K, x0.shape, device=x0.device)
    return torch.where(keep, x0, noise)


def q_posterior_probs(x_t, x0, t_float, t_prev_float):
    """
    Compute P(x_{t-1} = 1 | x_t, x_0) for each pixel using the D3PM posterior.

    From D3PM paper eq. (3):
      q(x_{t-1} | x_t, x_0) proportional to Q_t[x_{t-1} -> x_t] * Q_bar_{t-1}[x_0 -> x_{t-1}]

    For K=2:
      Q_t[c -> x_t]          = (1 - beta_t) if c == x_t  else  beta_t / K
      Q_bar_{t-1}[x_0 -> c]  = (1 + ab_{t-1}) / 2  if c == x_0  else  (1 - ab_{t-1}) / 2

    x_t, x0: LongTensor [B, 1, H, W]
    Returns: FloatTensor [B, 1, H, W] = P(x_{t-1} = 1 | x_t, x_0)
    """
    ab_t    = cosine_alpha_bar(t_float).to(x_t.device)[:, None, None, None]
    ab_prev = cosine_alpha_bar(t_prev_float).to(x_t.device)[:, None, None, None]

    # Single-step signal retention: alpha_t = ab_t / ab_{t-1}
    alpha_t = (ab_t / ab_prev.clamp(min=1e-8)).clamp(0.0, 1.0)
    beta_t  = 1.0 - alpha_t  # per-step flip probability

    xt_f = x_t.float()  # 0.0 or 1.0 per pixel
    x0_f = x0.float()

    # Q_t[c -> x_t]: single-step probability of going FROM c TO x_t
    qt_c0 = (1.0 - beta_t) * (1.0 - xt_f) + (beta_t / K) * xt_f  # c=0
    qt_c1 = (beta_t / K) * (1.0 - xt_f) + (1.0 - beta_t) * xt_f  # c=1

    # Q_bar_{t-1}[x_0 -> c]: cumulative probability of going FROM x_0 TO c after t-1 steps
    qbar_c0 = (1.0 + ab_prev) / 2.0 * (1.0 - x0_f) + (1.0 - ab_prev) / 2.0 * x0_f  # c=0
    qbar_c1 = (1.0 - ab_prev) / 2.0 * (1.0 - x0_f) + (1.0 + ab_prev) / 2.0 * x0_f  # c=1

    # Unnormalized posterior weights
    theta0 = qt_c0 * qbar_c0
    theta1 = qt_c1 * qbar_c1

    return theta1 / (theta0 + theta1 + 1e-8)  # P(x_{t-1} = 1 | x_t, x_0)

## Visualise the Forward Process

As t increases, pixels are increasingly randomised. Unlike continuous diffusion (which blurs to grey),
discrete diffusion produces salt-and-pepper noise — pixels remain either black or white, just increasingly shuffled.

In [ ]:
torch.manual_seed(0)
timesteps_to_show = [0.0, 0.1, 0.3, 0.5, 0.7, 1.0]
fig, axes = plt.subplots(2, len(timesteps_to_show), figsize=(12, 4))

for row, (x0_img, label) in enumerate([(black, 'black x₀'), (white, 'white x₀')]):
    x0 = x0_img.to(device)  # [1, 1, H, W]
    for col, t_val in enumerate(timesteps_to_show):
        if t_val == 0.0:
            xt = x0[0, 0].float().cpu()
        else:
            t_tensor = torch.tensor([t_val], device=device)
            xt = q_sample(x0, t_tensor)[0, 0].float().cpu()
        axes[row, col].imshow(xt.numpy(), cmap='gray', vmin=0, vmax=1, interpolation='nearest')
        axes[row, col].set_title(f't={t_val:.1f}', fontsize=9)
        axes[row, col].axis('off')
    axes[row, 0].set_ylabel(label, fontsize=10)

plt.suptitle('D3PM-Uniform Forward Process (K=2)', fontsize=12)
plt.tight_layout()
plt.show()

## Helper Functions

Same local consistency check and visualisation as the continuous notebook.

In [ ]:
def local_consistency_mask(binary_img):
    """
    For each pixel, check if the majority colour in its 3x3 periodic patch
    equals the centre pixel colour. Returns a boolean mask of violations.
    """
    binary_img = binary_img.cpu()
    h, w = binary_img.shape
    bad = torch.zeros((h, w), dtype=torch.bool)
    for i in range(h):
        rows = torch.arange(i - 1, i + 2) % h
        for j in range(w):
            cols = torch.arange(j - 1, j + 2) % w
            patch = binary_img[rows][:, cols]
            majority = 1 if patch.sum().item() >= 5 else 0
            bad[i, j] = (majority != binary_img[i, j].item())
    return bad


def draw_binary(ax, binary_img, violation_mask=None):
    img_np = binary_img.detach().cpu().float().numpy()
    ax.imshow(img_np, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    ax.set_xticks([x - 0.5 for x in range(binary_img.shape[1] + 1)], minor=True)
    ax.set_yticks([y - 0.5 for y in range(binary_img.shape[0] + 1)], minor=True)
    ax.grid(which='minor', color='#2b4ea2', linestyle='-', linewidth=0.4)
    if violation_mask is not None and bool(violation_mask.any()):
        overlay = torch.zeros((*binary_img.shape, 4), dtype=torch.float32)
        overlay[..., 0] = violation_mask.float()          # red channel
        overlay[..., 3] = violation_mask.float() * 0.45   # alpha
        ax.imshow(overlay.numpy())
    ax.set_xticks([])
    ax.set_yticks([])

## Discrete Diffusion CNN

**Architecture:** Same MinimalResNet as the continuous version (circular padding).

**Key difference:** 
- The network now predicts $\tilde{p}_\theta(x_0 = 1 \mid x_t, t)$ (probability of each pixel being white),
  rather than predicting the noise $\epsilon$.
- This is the **x₀-parameterisation** from D3PM paper Section 3.3.
- Training loss: binary cross-entropy against the true $x_0$.

**Sampling:** D3PM reverse process — at each step, marginalise the predicted $x_0$ over the posterior
$q(x_{t-1} \mid x_t, x_0)$ derived from the transition matrices.

In [ ]:
class DiscreteDiffusionCNN(nn.Module):
    """
    CNN for discrete (K=2) diffusion using the x0-parameterisation (D3PM Section 3.3).

    Input:  x_t normalised from {0,1} to {-1,+1}  (1 channel, same as continuous version)
    Output: single logit per pixel  →  sigmoid gives P(x0=1 | x_t, t)
    Loss:   binary cross-entropy against true x0
    """

    def __init__(self, H, emb_dim=256, num_layers=2):
        super().__init__()
        self.backbone = MinimalResNet(
            channels=1,
            emb_dim=emb_dim,
            mode='circular',
            conditional=False,
            num_layers=num_layers,
            normalization=None,
            lastksize=3,
        )

    def forward(self, t, x_t_normalized):
        return self.backbone(t, x_t_normalized)

    def predict_x0_probs(self, t, x_t):
        x_norm = x_t.float() * 2.0 - 1.0
        return torch.sigmoid(self.forward(t, x_norm))

    @torch.no_grad()
    def sample(self, nsteps=200, seed=None):
        if seed is not None:
            torch.manual_seed(seed)
        self.eval()
        x_t = torch.randint(0, K, (1, 1, H, W), device=device)

        for i in range(nsteps, 0, -1):
            t_float = torch.tensor([i / nsteps],       device=device)
            t_prev  = torch.tensor([(i - 1) / nsteps], device=device)

            p_x0_1 = self.predict_x0_probs(t_float, x_t)
            p_x0_0 = 1.0 - p_x0_1

            if i == 1:
                x_t = (p_x0_1 > 0.5).long()
            else:
                x0_zeros = torch.zeros_like(x_t)
                x0_ones  = torch.ones_like(x_t)
                p_prev_given_x0_0 = q_posterior_probs(x_t, x0_zeros, t_float, t_prev)
                p_prev_given_x0_1 = q_posterior_probs(x_t, x0_ones,  t_float, t_prev)
                p_prev_1 = p_x0_0 * p_prev_given_x0_0 + p_x0_1 * p_prev_given_x0_1
                x_t = torch.bernoulli(p_prev_1).long()

        return x_t


print('DiscreteDiffusionCNN class defined.')

In [ ]:
# ── Verify circular padding ───────────────────────────────────────────────────
# padding='same' + padding_mode='circular' can silently fall back to zero
# padding in some PyTorch versions. This test detects that.
#
# Setup: uniform averaging filter (3x3, all weights = 1/9).
#        Input: top row = 1, everything else = 0.
#        With CIRCULAR padding the bottom row of the output should receive
#        contribution from the top row (toroidal wrap) → bottom_val > 0.
#        With ZERO padding the bottom row sees zeros on both sides → bottom_val = 0.

with torch.no_grad():
    _conv = nn.Conv2d(1, 1, kernel_size=3, padding='same',
                      padding_mode='circular', bias=False)
    _conv.weight.data.fill_(1.0 / 9.0)
    _img = torch.zeros(1, 1, 8, 8)
    _img[0, 0, 0, :] = 1.0          # top row white, rest black
    _out = _conv(_img)
    top_val    = _out[0, 0,  0, 0].item()
    bottom_val = _out[0, 0, -1, 0].item()

print('Circular padding verification (3x3 averaging filter, top row = 1):')
print(f'  Top row output:    {top_val:.4f}  '
      f'(circular → 3/9=0.333 | zero → 2/9=0.222)')
print(f'  Bottom row output: {bottom_val:.4f}  '
      f'(circular → 1/9=0.111 | zero → 0/9=0.000)')

if bottom_val > 0.05:
    print('✓  Circular padding is working correctly.')
else:
    print('✗  WARNING: circular padding NOT active — zero padding is being used!')

## Training

Training objective: the **auxiliary cross-entropy loss** from D3PM (eq. 5 of Austin et al.).

For each batch:
1. Sample a random timestep $t$.
2. Corrupt $x_0 \to x_t$ using the forward process.
3. Predict $\tilde{p}_\theta(x_0 \mid x_t, t)$ with the CNN.
4. Minimise $-\log \tilde{p}_\theta(x_0 \mid x_t)$ (binary cross-entropy).

In [ ]:
def train_discrete_diffusion(model, dataset, num_epochs=300, lr=2e-4, nsteps=T):
    sampler   = RandomSampler(dataset, replacement=True, num_samples=20000)
    loader    = DataLoader(dataset, batch_size=64, sampler=sampler)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    final_loss = 0.0

    for epoch in range(num_epochs):
        total_loss = 0.0
        for x0_batch, _ in loader:
            x0_batch = x0_batch.to(device)
            B = x0_batch.shape[0]
            t_int   = torch.randint(1, nsteps + 1, (B,), device=device)
            t_float = t_int.float() / nsteps
            x_t     = q_sample(x0_batch, t_float)
            x_norm  = x_t.float() * 2.0 - 1.0
            logits  = model(t_float, x_norm)
            loss    = F.binary_cross_entropy_with_logits(logits, x0_batch.float())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        final_loss = total_loss / len(loader)
        if (epoch + 1) % 50 == 0:
            print(f'  Epoch {epoch+1:>3}/{num_epochs}  loss={final_loss:.4f}')

    model.eval()
    return model, final_loss


print('Training function defined.')

## Generate Samples and Evaluate Local Consistency

Same evaluation as the continuous notebook:
- **Many steps** (400): slow, careful denoising → should produce locally consistent images.
- **Few steps** (10): rushed denoising → more violations (red pixels).

A pixel is a **violation** if the majority colour in its 3×3 periodic patch differs from its own colour.

In [ ]:
training_imgs = [
    torch.zeros((H, W), dtype=torch.int),
    torch.ones( (H, W), dtype=torch.int),
]


def plot_denoising_process(model, nsteps, title='', n_snapshots=8, seed=None):
    """
    Run the reverse process once and plot snapshots at evenly-spaced timesteps.
    For short runs (nsteps <= 10) every step is shown.
    Violations are highlighted in red on the final frame only.
    """
    if seed is not None:
        torch.manual_seed(seed)
    model.eval()

    x_t = torch.randint(0, K, (1, 1, H, W), device=device)

    # Decide which steps to snapshot (always include the very first and last)
    if nsteps <= 10:
        capture = set(range(nsteps, -1, -1))
    else:
        step_size = max(1, nsteps // (n_snapshots - 1))
        capture = set(range(nsteps, 0, -step_size)) | {1}

    snapshots = [(nsteps, x_t[0, 0].cpu().clone())]  # initial noise

    for i in range(nsteps, 0, -1):
        t_float = torch.tensor([i / nsteps],       device=device)
        t_prev  = torch.tensor([(i - 1) / nsteps], device=device)

        with torch.no_grad():
            p_x0_1 = model.predict_x0_probs(t_float, x_t)
            p_x0_0 = 1.0 - p_x0_1
            if i == 1:
                x_t = (p_x0_1 > 0.5).long()
            else:
                x0_z = torch.zeros_like(x_t)
                x0_o = torch.ones_like(x_t)
                p_prev_1 = (p_x0_0 * q_posterior_probs(x_t, x0_z, t_float, t_prev)
                          + p_x0_1 * q_posterior_probs(x_t, x0_o, t_float, t_prev))
                x_t = torch.bernoulli(p_prev_1).long()

        if i in capture:
            snapshots.append((i - 1, x_t[0, 0].cpu().clone()))

    # Thin to at most n_snapshots frames
    if len(snapshots) > n_snapshots:
        idx = np.linspace(0, len(snapshots) - 1, n_snapshots, dtype=int)
        snapshots = [snapshots[i] for i in idx]

    n = len(snapshots)
    fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 2.8))
    if n == 1:
        axes = [axes]

    for ax, (step, img) in zip(axes, snapshots):
        is_final = (step == 0)
        mask = local_consistency_mask(img) if is_final else None
        draw_binary(ax, img, mask)
        label = f't={step}' + (' (final)' if is_final else '')
        ax.set_title(label, fontsize=8)

    fig.suptitle(title, fontsize=10, y=1.01)
    plt.tight_layout()
    return fig


print('Helpers defined.')

In [ ]:
log_path   = 'results_log.csv'
fieldnames = ['num_layers', 'emb_dim', 'image_size', 'receptive_field', 'n_params',
              'nsteps_valid', 'nsteps_invalid', 'final_loss',
              'valid_violations_mean', 'invalid_violations_mean',
              'valid_violations_all', 'invalid_violations_all']

# Read already-completed depths so we never redo work
already_done = set()
if os.path.exists(log_path):
    with open(log_path) as f:
        for row in csv.DictReader(f):
            already_done.add(int(row['num_layers']))
    print(f'Already logged NUM_LAYERS: {sorted(already_done)} — will skip these')

write_header = not os.path.exists(log_path)

for NUM_LAYERS in NUM_LAYERS_LIST:
    if NUM_LAYERS in already_done:
        print(f'Skipping NUM_LAYERS={NUM_LAYERS} (already in log)')
        continue

    RF = 2 * (NUM_LAYERS + 2) + 1
    print(f"\n{'='*60}")
    print(f"Experiment  NUM_LAYERS={NUM_LAYERS}  RF={RF}x{RF}  "
          f"({RF*RF} pixels, {100*RF*RF//(H*W)}% of grid)")
    print(f"{'='*60}")

    # ── Build model ────────────────────────────────────────────
    model = DiscreteDiffusionCNN(H=H, emb_dim=EMB_DIM, num_layers=NUM_LAYERS).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f'Parameters: {n_params:,}')

    # ── Train ──────────────────────────────────────────────────
    model, final_loss = train_discrete_diffusion(
        model, dataset, num_epochs=300, lr=2e-4, nsteps=T)
    print(f'Training complete. Final loss: {final_loss:.4f}')

    # ── Generate & evaluate ────────────────────────────────────
    valid_samples   = [model.sample(nsteps=NSTEPS_VALID)[0, 0].cpu()   for _ in range(4)]
    invalid_samples = [model.sample(nsteps=NSTEPS_INVALID)[0, 0].cpu() for _ in range(4)]
    valid_masks     = [local_consistency_mask(img) for img in valid_samples]
    invalid_masks   = [local_consistency_mask(img) for img in invalid_samples]

    valid_counts   = [int(m.sum()) for m in valid_masks]
    invalid_counts = [int(m.sum()) for m in invalid_masks]
    print(f'Valid violations   ({NSTEPS_VALID} steps):   {valid_counts}  mean={np.mean(valid_counts):.2f}')
    print(f'Invalid violations ({NSTEPS_INVALID} steps): {invalid_counts}  mean={np.mean(invalid_counts):.2f}')

    # ── Sample overview plot ───────────────────────────────────
    fig = plt.figure(figsize=(16, 4))
    outer = GridSpec(1, 3, width_ratios=[2, 4, 4], wspace=0.15)

    left = outer[0].subgridspec(1, 2, wspace=0.1)
    ax_a0 = fig.add_subplot(left[0, 0])
    ax_a1 = fig.add_subplot(left[0, 1])
    draw_binary(ax_a0, training_imgs[0])
    draw_binary(ax_a1, training_imgs[1])
    ax_a0.set_title('Training Data', fontsize=12, pad=10)

    mid = outer[1].subgridspec(1, 4, wspace=0.1)
    mid_axes = [fig.add_subplot(mid[0, i]) for i in range(4)]
    for ax, img in zip(mid_axes, valid_samples):
        draw_binary(ax, img)
    mid_axes[1].set_title(f'Valid Samples ({NSTEPS_VALID} steps)', fontsize=11, pad=10)

    right = outer[2].subgridspec(1, 4, wspace=0.1)
    right_axes = [fig.add_subplot(right[0, i]) for i in range(4)]
    for ax, img, mask in zip(right_axes, invalid_samples, invalid_masks):
        draw_binary(ax, img, mask)
    right_axes[1].set_title(f'Invalid Samples ({NSTEPS_INVALID} steps)', fontsize=11, pad=10)

    fig.suptitle(f'CNN  L={NUM_LAYERS}  RF={RF}×{RF}  |  '
                 f'valid mean={np.mean(valid_counts):.1f}  '
                 f'invalid mean={np.mean(invalid_counts):.1f}', fontsize=12, y=1.02)
    plt.tight_layout()
    fname = f'toy_discrete_L{NUM_LAYERS}_valid{NSTEPS_VALID}_invalid{NSTEPS_INVALID}.png'
    plt.savefig(fname, dpi=180, bbox_inches='tight')
    print(f'Saved {fname}')
    plt.show()

    # ── Intermediate denoising plots ───────────────────────────
    # Valid: 8 evenly-spaced snapshots across 400 steps
    fig_v = plot_denoising_process(
        model, nsteps=NSTEPS_VALID, n_snapshots=8,
        title=f'CNN L={NUM_LAYERS}  —  denoising process ({NSTEPS_VALID} steps)')
    fname_v = f'toy_discrete_L{NUM_LAYERS}_denoise_valid.png'
    fig_v.savefig(fname_v, dpi=180, bbox_inches='tight')
    print(f'Saved {fname_v}')
    plt.show()

    # Invalid: all steps shown (only 3 steps total)
    fig_i = plot_denoising_process(
        model, nsteps=NSTEPS_INVALID, n_snapshots=NSTEPS_INVALID + 1,
        title=f'CNN L={NUM_LAYERS}  —  denoising process ({NSTEPS_INVALID} steps)')
    fname_i = f'toy_discrete_L{NUM_LAYERS}_denoise_invalid.png'
    fig_i.savefig(fname_i, dpi=180, bbox_inches='tight')
    print(f'Saved {fname_i}')
    plt.show()

    # ── Log ────────────────────────────────────────────────────
    row = {
        'num_layers':              NUM_LAYERS,
        'emb_dim':                 EMB_DIM,
        'image_size':              H,
        'receptive_field':         RF,
        'n_params':                n_params,
        'nsteps_valid':            NSTEPS_VALID,
        'nsteps_invalid':          NSTEPS_INVALID,
        'final_loss':              round(final_loss, 4),
        'valid_violations_mean':   round(np.mean(valid_counts),   2),
        'invalid_violations_mean': round(np.mean(invalid_counts), 2),
        'valid_violations_all':    str(valid_counts),
        'invalid_violations_all':  str(invalid_counts),
    }
    with open(log_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
            write_header = False
        writer.writerow(row)
    already_done.add(NUM_LAYERS)

print(f"\n{'='*60}")
print(f"All experiments done. Results in {log_path}")
print(f"{'='*60}")
with open(log_path) as f:
    for line in f:
        print(line, end='')

## TODO: Graph Extension (next steps)

Once the theoretical derivation is available from the theory team, the following extensions can be implemented:

1. **Node-feature graphs** — treat each node's categorical feature as a discrete variable;
   replace the CNN with a GNN using local message passing.
2. **Edge prediction** — add binary edge states {present, absent} to the discrete state space.
3. **Joint node+edge diffusion** — full molecule graph generation (QM9 dataset).

The discrete forward/reverse process defined here (D3PM-uniform) carries over directly
to graphs: the transition matrix $Q_t$ is applied independently to each node feature (and edge),
and the GNN replaces the CNN as the denoising network.